# EDA



## Startup cells

In [0]:
# Set environment variables for sagemaker_studio imports

import os
os.environ['DataZoneProjectId'] = '5soog4ogof43l5'
os.environ['DataZoneDomainId'] = 'dzd-6vrgvadzc0959l'
os.environ['DataZoneEnvironmentId'] = '4fmbqiqxot49ex'
os.environ['DataZoneDomainRegion'] = 'us-east-2'

# create both a function and variable for metadata access
_resource_metadata = None

def _get_resource_metadata():
    global _resource_metadata
    if _resource_metadata is None:
        _resource_metadata = {
            "AdditionalMetadata": {
                "DataZoneProjectId": "5soog4ogof43l5",
                "DataZoneDomainId": "dzd-6vrgvadzc0959l",
                "DataZoneEnvironmentId": "4fmbqiqxot49ex",
                "DataZoneDomainRegion": "us-east-2",
            }
        }
    return _resource_metadata
metadata = _get_resource_metadata()

In [0]:
"""
Logging Configuration

Purpose:
--------
This sets up the logging framework for code executed in the user namespace.
"""

from typing import Optional


def _set_logging(log_dir: str, log_file: str, log_name: Optional[str] = None):
    import os
    import logging
    from logging.handlers import RotatingFileHandler

    level = logging.INFO
    max_bytes = 5 * 1024 * 1024
    backup_count = 5

    # fallback to /tmp dir on access, helpful for local dev setup
    try:
        os.makedirs(log_dir, exist_ok=True)
    except Exception:
        log_dir = "/tmp/kernels/"

    os.makedirs(log_dir, exist_ok=True)
    log_path = os.path.join(log_dir, log_file)

    logger = logging.getLogger() if not log_name else logging.getLogger(log_name)
    logger.handlers = []
    logger.setLevel(level)

    formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")

    # Rotating file handler
    fh = RotatingFileHandler(filename=log_path, maxBytes=max_bytes, backupCount=backup_count, encoding="utf-8")
    fh.setFormatter(formatter)
    logger.addHandler(fh)

    logger.info(f"Logging initialized for {log_name}.")


_set_logging("/var/log/computeEnvironments/kernel/", "kernel.log")
_set_logging("/var/log/studio/data-notebook-kernel-server/", "metrics.log", "metrics")

In [0]:
import logging
from sagemaker_studio import ClientConfig, sqlutils, sparkutils, dataframeutils

logger = logging.getLogger(__name__)
logger.info("Initializing sparkutils")
spark = sparkutils.init()
logger.info("Finished initializing sparkutils")

In [0]:
def _reset_os_path():
    """
    Reset the process's working directory to handle mount timing issues.
    
    This function resolves a race condition where the Python process starts
    before the filesystem mount is complete, causing the process to reference
    old mount paths and inodes. By explicitly changing to the mounted directory
    (/home/sagemaker-user), we ensure the process uses the correct, up-to-date
    mount point.
    
    The function logs stat information (device ID and inode) before and after
    the directory change to verify that the working directory is properly
    updated to reference the new mount.
    
    Note:
        This is executed at module import time to ensure the fix is applied
        as early as possible in the kernel initialization process.
    """
    try:
        import os
        import logging

        logger = logging.getLogger(__name__)
        logger.info("---------Before------")
        logger.info("CWD: %s", os.getcwd())
        logger.info("stat('.'): %s %s", os.stat('.').st_dev, os.stat('.').st_ino)
        logger.info("stat('/home/sagemaker-user'): %s %s", os.stat('/home/sagemaker-user').st_dev, os.stat('/home/sagemaker-user').st_ino)

        os.chdir("/home/sagemaker-user")

        logger.info("---------After------")
        logger.info("CWD: %s", os.getcwd())
        logger.info("stat('.'): %s %s", os.stat('.').st_dev, os.stat('.').st_ino)
        logger.info("stat('/home/sagemaker-user'): %s %s", os.stat('/home/sagemaker-user').st_dev, os.stat('/home/sagemaker-user').st_ino)
    except Exception as e:
        logger.exception(f"Failed to reset working directory: {e}")

_reset_os_path()

## Notebook

# Explanatory Data Analysis on UCF Parking Data

# 1

## Introduction

### Project Description

**Goal/Purpose:** 

This project, parkUCF, aims to facilitate student parking at the University of Central Florida's parking garages. The objective is to deploy an application that leverages machine learning predictions to display information and insights to students regarding the expected garage occupancy for the day. This notebook will outline and document the process of explanatory data analysis, and its findings will be instrumental to the completion of parkUCF.

<p>&nbsp;</p>

**Questions to be Answered:**

- Do time formatting issues cause problems in the dataset?
- What are the busiest and least busy garages and days of the week?
- How similar are occupancy trends from day to day?

<p>&nbsp;</p>

**Assumptions/Methodology/Scope:** 

Assumptions:

- All timestamps are timezone-aware/in UTC
- The device which scans license plates returns accurate vehicle counts
- The number of occupied spots represents the number of vehicles inside the garage

Methodology/processing steps:

- Data inspection, cleaning and preprocessing
- Multivariate analysis
- Feature engineering
- Visualization

Scope:

- Exploring and understanding the dataset
- Only the data provided is analyzed

<p>&nbsp;</p>

### Data Description

**Content:** 

This dataset is a JSON file of a constantly increasing amount of data points which contains parking garage data from 6 UCF parking garages (garages A, B, C, D, H, and I). 

<p>&nbsp;</p>

**Description of Attributes:** 

| Column  | Description |
| :------ | :---------- |
| name    | Name of the garage|
|available| total - amount of vehicles in garage|
|occupied | Amount of vehicles in garage|
|total    | Total amount of spots in garage |
|occupancy_rate|occupied/available|
|event_reserved|Amount of spots reserved for an event|
|event_name|The name of the event|
|timestamp| The timestamp in UTC of the datapoint|

<p>&nbsp;</p>

# 2

## Acquiring and Loading Data

### Importing Libraries and Notebook Setup

In [0]:
# Data manipulation
import datetime
import numpy as np
import pandas as pd
import pandas.api.types as ptypes
!pip install plotly
import plotly
!pip install awswrangler
import awswrangler as wr
!pip install skimpy
from skimpy import skim, clean_columns
!pip install prophet
from prophet import Prophet
from prophet.plot import plot_plotly, plot_components_plotly
!pip install supabase
from supabase import create_client, Client
import boto3
import json

# Visualizations
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Pandas settings
pd.options.display.max_columns = None
pd.options.display.max_colwidth = 60
pd.options.display.float_format = '{:,.3f}'.format

# Visualization settings
from matplotlib import rcParams
plt.style.use('fivethirtyeight')
rcParams['figure.figsize'] = (16, 5)   
rcParams['axes.spines.right'] = False
rcParams['axes.spines.top'] = False
rcParams['font.size'] = 12
# rcParams['figure.dpi'] = 300
rcParams['savefig.dpi'] = 300
plt.rc('xtick', labelsize=11)
plt.rc('ytick', labelsize=11)
custom_palette = ['#003f5c', '#444e86', '#955196', '#dd5182', '#ff6e54', '#ffa600']
custom_hue = ['#004c6d', '#346888', '#5886a5', '#7aa6c2', '#9dc6e0', '#c1e7ff']
custom_divergent = ['#00876c', '#6aaa96', '#aecdc2', '#f1f1f1', '#f0b8b8', '#e67f83', '#d43d51']
sns.set_palette(custom_palette)
%config InlineBackend.figure_format = 'retina'

  Using cached skimpy-0.0.19-py3-none-any.whl.metadata (33 kB)


  Using cached numpy-2.3.5-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)


  Using cached pandas_stubs-2.3.3.251219-py3-none-any.whl.metadata (10 kB)


  Using cached typeguard-4.4.4-py3-none-any.whl.metadata (3.3 kB)


  Using cached pydoclint-0.8.3-py3-none-any.whl.metadata (10 kB)


...

  Using cached h2-4.3.0-py3-none-any.whl.metadata (5.1 kB)
  Using cached hyperframe-6.1.0-py3-none-any.whl.metadata (4.3 kB)


  Using cached hpack-4.1.0-py3-none-any.whl.metadata (4.6 kB)


Using cached supabase-2.27.0-py3-none-any.whl (16 kB)
Using cached postgrest-2.27.0-py3-none-any.whl (21 kB)
Using cached realtime-2.27.0-py3-none-any.whl (22 kB)
Using cached storage3-2.27.0-py3-none-any.whl (27 kB)
Using cached supabase_auth-2.27.0-py3-none-any.whl (48 kB)
Using cached supabase_functions-2.27.0-py3-none-any.whl (8.5 kB)
Using cached h2-4.3.0-py3-none-any.whl (61 kB)
Using cached hpack-4.1.0-py3-none-any.whl (34 kB)
Using cached hyperframe-6.1.0-py3-none-any.whl (13 kB)
Using cached deprecation-2.1.0-py2.py3-none-any.whl (11 kB)
Using cached StrEnum-0.4.15-py3-none-any.whl (8.9 kB)


   ━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━  6/11 [supabase-functions]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [supabase]


### Loading Data

In [0]:
s3_path = "s3://parkucfdatalake/raw/"
df = wr.s3.read_json(path=s3_path)
df = df.sort_values(by='timestamp', ascending=True)
df = df[['name', 'available', 'occupied', 'total', 'occupancy_rate', 'event_reserved', 'event_name', 'timestamp']]

### Basic Data Exploration

In [0]:
print(f"Rows count: {df.shape[0]}\nColumns count: {df.shape[1]}")

Rows count: 221014
Columns count: 8


In [0]:
df.head()

,name,available,occupied,total,occupancy_rate,event_reserved,event_name,timestamp
0,Garage A,"1,647.000",0.000,"1,647.000",0.000,NaN,None,2025-09-22 01:45:17.937
3,Garage H,"1,340.000",0.000,"1,340.000",0.000,NaN,None,2025-09-22 01:45:17.937
1,Garage B,"1,289.000",0.000,"1,289.000",0.000,NaN,None,2025-09-22 01:45:17.937
2,Garage C,"1,852.000",0.000,"1,852.000",0.000,NaN,None,2025-09-22 01:45:17.937
7,Garage H,"1,340.000",0.000,"1,340.000",0.000,NaN,None,2025-09-22 01:47:17.570


In [0]:
df.tail()

,name,available,occupied,total,occupancy_rate,event_reserved,event_name,timestamp
2,Garage C,"1,850.000",2.000,"1,852.000",0.001,0.000,NaN,2025-12-20 06:15:16.572
3,Garage D,"1,263.000",16.000,"1,289.000",0.012,0.000,NaN,2025-12-20 06:15:16.572
4,Garage H,"1,340.000",0.000,"1,340.000",0.000,0.000,NaN,2025-12-20 06:15:16.572
5,Garage I,"1,270.000",0.000,"1,270.000",0.000,0.000,NaN,2025-12-20 06:15:16.572
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT


In [0]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 221014 entries, 0 to 0
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   name            221013 non-null  object        
 1   available       221013 non-null  float64       
 2   occupied        221013 non-null  float64       
 3   total           221013 non-null  float64       
 4   occupancy_rate  221013 non-null  float64       
 5   event_reserved  214410 non-null  float64       
 6   event_name      13459 non-null   object        
 7   timestamp       221013 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(5), object(2)
memory usage: 15.2+ MB


- `name`,`event_name` are **strings**.
- `available`, `occupied`, `total`, `event_reserved` are **integers**.
- `occupancy_rate` is a **float**.
- `timestamp` is a **datetime** object.

### Check Missing Data

In [0]:
missing_percent = df.isna().mean().sort_values(ascending=False)
print('---- Percentage of Missing Values (%) -----')
if missing_percent.sum():
    print(missing_percent[missing_percent > 0] * 100)
else:
    print('None')

---- Percentage of Missing Values (%) -----
event_name       93.910
event_reserved    2.988
name              0.000
available         0.000
total             0.000
occupied          0.000
occupancy_rate    0.000
timestamp         0.000
dtype: float64


### Check for Duplicate Rows

In [0]:
print(f"No. of entirely duplicated rows: {df.duplicated().sum()}")

df = df[~df.duplicated()]
df = df.reset_index(drop=True)

No. of entirely duplicated rows: 6282


### Check Uniqueness of Data

In [0]:
# Print the percentage similarity of values (the lower %, the better)
num_unique = df.nunique().sort_values()
print('---- Percentage Similarity of Values (%) -----')
print(100/num_unique)

---- Percentage Similarity of Values (%) -----
name             16.667
event_reserved    9.091
event_name        5.556
total             5.556
occupancy_rate    0.067
available         0.054
occupied          0.050
timestamp         0.003
dtype: float64


### Check Data Range

In [0]:
# Print summary statistics
df.describe(include='all')
skim(df)

╭──────────────────────────────────────────────── skimpy summary ─────────────────────────────────────────────────╮
│          Data Summary                Data Types                                                                 │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓ ┏━━━━━━━━━━━━━┳━━━━━━━┓                                                          │
│ ┃ Dataframe         ┃ Values ┃ ┃ Column Type ┃ Count ┃                                                          │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩ ┡━━━━━━━━━━━━━╇━━━━━━━┩                                                          │
│ │ Number of rows    │ 214732 │ │ float64     │ 5     │                                                          │
│ │ Number of columns │ 8      │ │ string      │ 2     │                                                          │
│ └───────────────────┴────────┘ │ datetime64  │ 1     │                                                          │
│                                └─────────────┴───────┘                                                          │
│                                                     number                                                      │
│ ┏━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━━┓  │
│ ┃ column         ┃ NA   ┃ NA %             ┃ mean   ┃ sd     ┃ p0   ┃ p25   ┃ p50   ┃ p75   ┃ p100  ┃ hist   ┃  │
│ ┡━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━━┩  │
│ │ available      │    1 │ 0.00046569677551 │   1003 │  574.9 │    0 │   471 │  1257 │  1369 │  1852 │ ▄▄▂▂█▅ │  │
│ │                │      │           552634 │        │        │      │       │       │       │       │        │  │
│ │ occupied       │    1 │ 0.00046569677551 │  461.7 │    514 │    0 │     2 │   180 │   924 │  2038 │ █▁▃▂▁  │  │
│ │                │      │           552634 │        │        │      │       │       │       │       │        │  │
│ │ total          │    1 │ 0.00046569677551 │   1460 │  225.6 │ 1209 │  1289 │  1340 │  1647 │  1852 │ █▃  ▃▃ │  │
│ │                │      │           552634 │        │        │      │       │       │       │       │        │  │
│ │ occupancy_rate │    1 │ 0.00046569677551 │ 0.3303 │ 0.3727 │    0 │ 0.001 │ 0.117 │ 0.666 │ 1.521 │  █▁▃▂  │  │
│ │                │      │           552634 │        │        │      │       │       │       │       │        │  │
│ │ event_reserved │ 6274 │ 2.92178156958441 │  1.718 │   9.27 │    0 │     0 │     0 │     0 │   100 │   █    │  │
│ │                │      │               23 │        │        │      │       │       │       │       │        │  │
│ └────────────────┴──────┴──────────────────┴────────┴────────┴──────┴───────┴───────┴───────┴───────┴────────┘  │
│                                                    datetime                                                     │
│ ┏━━━━━━━━━━━━━┳━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓  │
│ ┃ column      ┃ NA  ┃ NA %                    ┃ first                  ┃ last                   ┃ frequency  ┃  │
│ ┡━━━━━━━━━━━━━╇━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩  │
│ │ timestamp   │   1 │  0.00046569677551552634 │       2025-09-22       │       2025-12-20       │ None       │  │
│ │             │     │                         │    01:45:17.937000     │    06:15:16.572000     │            │  │
│ └─────────────┴─────┴─────────────────────────┴────────────────────────┴────────────────────────┴────────────┘  │
│                                                     string                                                      │
│ ┏━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┓  │
│ ┃          ┃        ┃          ┃          ┃          ┃          ┃          ┃ chars    ┃ words    ┃ total     ┃  │
│ ┃ column   ┃ NA     ┃ NA %     ┃ shortest ┃ longest  ┃

### Areas to Fix

**Duplicate Rows**
- I am unsure why this level of duplicate rows exist, but this will be fixed 

**Irrelevant Columns**
- `event_name` and `event_reserved` are not needed.

**Redundant Columns**
- `occupied`, `available`, and `total` are redundant since we've already added `occupancy_rate` during extraction.

**Timestamp Format Inconsistency**
- The timezones in our dataset are not all in the same format, though they are all timezone-aware/in UTC. As the Facebook Prophet documentation explains, the format should ideally be `YYYY-MM-DD HH:MM:SS`.

# 3

## Data Preprocessing

### Remove Duplicate Rows

In [0]:
df = df[~df.duplicated()]
df

,name,available,occupied,total,occupancy_rate,event_reserved,event_name,timestamp
0,Garage A,"1,647.000",0.000,"1,647.000",0.000,NaN,None,2025-09-22 01:45:17.937
1,Garage H,"1,340.000",0.000,"1,340.000",0.000,NaN,None,2025-09-22 01:45:17.937
2,Garage B,"1,289.000",0.000,"1,289.000",0.000,NaN,None,2025-09-22 01:45:17.937
3,Garage C,"1,852.000",0.000,"1,852.000",0.000,NaN,None,2025-09-22 01:45:17.937
4,Garage H,"1,340.000",0.000,"1,340.000",0.000,NaN,None,2025-09-22 01:47:17.570
...,...,...,...,...,...,...,...,...
214727,Garage C,"1,850.000",2.000,"1,852.000",0.001,0.000,NaN,2025-12-20 06:15:16.572
214728,Garage D,"1,263.000",16.000,"1,289.000",0.012,0.000,NaN,2025-12-20 06:15:16.572
214729,Garage H,"1,340.000",0.000,"1,340.000",0.000,0.000,NaN,2025-12-20 06:15:16.572
214730,Garage I,"1,270.000",0.000,"1,270.000",0.000,0.000,NaN,2025-12-20 06:15:16.572


### Remove Redundant and Irrelevant Columns

In [0]:
cols_to_drop = ['event_name', 'event_reserved', 'occupied', 'available', 'total']
df = df.drop(columns=cols_to_drop, axis=1)

In [0]:
assert all(col not in df.columns for col in cols_to_drop)

### Fix Timestamp Formats

In [0]:
df['timestamp'] = pd.to_datetime(df['timestamp']).dt.tz_localize('UTC')
df['timestamp'] = df['timestamp'].dt.tz_convert('America/New_York')

### Metrics

In [0]:
mask = (df['timestamp'] >= '2025-09-22') & (df['timestamp'] <= '2025-12-06')
valid_df = df.loc[mask].copy()

valid_df['day'] = valid_df['timestamp'].dt.day_name()
valid_df['hour'] = valid_df['timestamp'].dt.hour
valid_df['minute'] = valid_df['timestamp'].dt.minute

avg_hourly_occupancy = (valid_df.groupby(['hour', 'minute'])['occupancy_rate'].mean())
h, m = avg_hourly_occupancy.idxmax()
busiest_time_value = avg_hourly_occupancy.max()

avg_daily_occupancy = (valid_df.groupby('day')['occupancy_rate'].mean())
busiest_day = avg_daily_occupancy.idxmax()
busiest_day_value = avg_daily_occupancy.max()

avg_garage_occupancy = (valid_df.groupby('name')['occupancy_rate'].mean())
busiest_garage = avg_garage_occupancy.idxmax()
busiest_garage_value = avg_garage_occupancy.max()

max_idx = valid_df['occupancy_rate'].idxmax()
max_row = valid_df.loc[max_idx]
max_time_str = f"{max_row['name']}, {max_row['timestamp'].strftime('%A %B %d %I:%M%p')}"
max_occupancy = max_row['occupancy_rate']

suffix = "PM" if h >= 12 else "AM"
display_h = h if h <= 12 else h - 12
if display_h == 0: display_h = 12
busiest_time_str = f"{display_h}:{m:02d} {suffix}"

percent_over_1 = (valid_df['occupancy_rate'] > 1).mean() * 100

metrics = {
    'time': busiest_time_str,
    'timeval': f"{busiest_time_value:.0%}",
    'day': busiest_day,
    'dayval': f"{busiest_day_value:.0%}",
    'garage': busiest_garage,
    'garageval': f"{busiest_garage_value:.0%}",
    'maxsum': max_time_str,
    'maxval': f"{max_occupancy:.0%}",
    'poc': f"{percent_over_1/100:.0%}",
    'one': 1
}
wr.s3.to_json(
    df=pd.DataFrame([metrics]),
    path="s3://parkucfdatalake/metrics/metrics.json",
    orient='records',
    indent=2
)
metrics

{'time': '9:18 AM',
 'timeval': '63%',
 'day': 'Thursday',
 'dayval': '42%',
 'garage': 'Garage I',
 'garageval': '68%',
 'maxsum': 'Garage H, Wednesday November 19 12:15PM',
 'maxval': '136%',
 'poc': '4%',
 'one': 1}

### More Preprocessing

In [0]:
columns_to_rename = {'timestamp': 'ds', 'occupancy_rate': 'y'}
df.rename(columns=columns_to_rename, inplace=True)

In [0]:
for char in ['A', 'B', 'C', 'D', 'H', 'I']:
    subset = df[df['name'].str[7] == char].copy()
    subset.drop(columns='name', axis=1)
    s3_processed_path = f's3://parkucfdatalake/processed/{char.lower()}/data.json'
    wr.s3.to_json(
        df=subset,
        path=s3_processed_path,
        orient='records',
        indent=2
    )

### Modeling

In [0]:
garages = ['a', 'b', 'c', 'd', 'h', 'i']
for g in garages:
    m = Prophet(
        growth="flat",
        daily_seasonality=False,
        weekly_seasonality=False,
        yearly_seasonality=False,
        changepoint_prior_scale=0.05,
        seasonality_prior_scale=10,
        seasonality_mode="additive",
    )
    m.add_seasonality(
        name="daily",
        period=1,
        fourier_order=10
    )
    m.add_seasonality(
        name="weekly",
        period=7,
        fourier_order=3
    )

    s3_path = f's3://parkucfdatalake/processed/{g}/'
    init = wr.s3.read_json(path=s3_path + "data.json")
    init['ds'] = pd.to_datetime(init['ds'], unit='ms')
    rs = init.set_index('ds')
    dfg = rs['y'].resample('15min').mean().reset_index()

    if g == 'd':
        dfg = dfg[~((dfg['ds'] >= '2025-10-04') & (dfg['ds'] <= '2025-10-21'))]

    m.fit(dfg)
    future = m.make_future_dataframe(periods=5, freq='15min')
    fcst = m.predict(future)
    fcst = fcst[(fcst['ds'] >= '2025-12-08') & (fcst['ds'] <= '2025-12-15')]
    daily = fcst.copy()
    daily = daily[len(daily)-96:len(daily)]
    if g != 'i':
        fcst = fcst[['ds', 'yhat_upper']]
        fcst = fcst.rename(columns={'ds': 'x', 'yhat_upper': 'y'})
    else:
        fcst = fcst[['ds', 'yhat']]
        fcst = fcst.rename(columns={'ds': 'x', 'yhat': 'y'})
    daily = daily[['ds', 'daily']]
    daily = daily.rename(columns={'ds': 'x', 'daily': 'y'})
    fcst['garage'] = g
    daily['garage'] = g
    fcst['id'] = f"{char.lower()}-" + fcst.index.astype(str)
    daily['id'] = f"{char.lower()}-" + daily.index.astype(str)
    sns.lineplot(data=fcst, x="x", y="y")
    wr.s3.to_json(
            df=daily,
            path=s3_path + "daily.json",
            orient='records',
            indent=2,
            date_format='iso'
    )
    wr.s3.to_json(
            df=fcst,
            path=s3_path + "weekly.json",
            orient='records',
            indent=2,
            date_format='iso'
    )

In [0]:
client = boto3.client("secretsmanager", region_name="us-east-2")
secret = client.get_secret_value(SecretId="supabase_service_role")
creds = json.loads(secret["SecretString"])

url: str = "https://zynijrbvoefmroitlcnn.supabase.co"
key: str = creds["service_role_key"]
supabase: Client = create_client(url, key)

metrics = wr.s3.read_json(path="s3://parkucfdatalake/metrics/metrics.json")
metrics = metrics.to_dict(orient='records')
supabase.table("metrics").upsert(metrics).execute() 

for garage in garages:
    daily_g = wr.s3.read_json(path=f"s3://parkucfdatalake/processed/{garage}/daily.json")
    daily_g['x'] = daily_g['x'].astype(str)
    daily_g = daily_g.to_dict(orient='records')
    supabase.table("daily_fcst").upsert(daily_g).execute()

    weekly_g = wr.s3.read_json(path=f"s3://parkucfdatalake/processed/{garage}/daily.json")
    weekly_g['x'] = weekly_g['x'].astype(str)
    weekly_g = weekly_g.to_dict(orient='records')
    supabase.table("weekly_fcst").upsert(weekly_g).execute()

## Shutdown cells

In [0]:
"""
Stop spark session and associated Athena Spark session
"""

from IPython import get_ipython as _get_ipython
_get_ipython().user_ns["spark"].stop()